In [5]:
from src.corrector.multiagentic_judge.run import EvaluationConfig, ModelConfig, Profile, IssueLedger, RoundResult, evaluate_round, check_convergence, IterativeEvaluationResult
from src.corrector.multiagentic_judge.configs import DEFAULT_RUBRIC

In [6]:
config = EvaluationConfig(
    model_config=ModelConfig(
        profiles={
            "factuality": Profile(
                family="openai",
                model="gpt-4o-mini",
                temperature=0.3,
            ),
            "linguistics": Profile(
                family="openai",
                model="gpt-4o-mini",
                temperature=0.3,
            ),
            "depth": Profile(
                family="openai",
                model="gpt-4o-mini",
                temperature=0.3,
            ),
            "reasoning": Profile(
                family="openai",
                model="gpt-4o-mini",
                temperature=0.3,
            ),
            "synthesis": Profile(
                family="openai",
                model="gpt-4o-mini",
                temperature=0.3,
            ),
        },
    ),
    rubric=DEFAULT_RUBRIC,
    min_rounds=1,
    max_rounds=1,
    convergence_score=4.3,
    convergence_variance=0.3,
)

In [7]:
# This output has a factual error (nucleus instead of chloroplast)
# and decent clarity — a good stress test for calibration.
first_output = (
    "Photosynthesis is the process by which plants convert sunlight into energy. "
    "It occurs in the nucleus of the cell, where chlorophyll captures light and "
    "combines water and carbon dioxide to produce glucose and oxygen."
)
prompt = "Explain photosynthesis to a high school student."


ledger = IssueLedger()
rounds: list[RoundResult] = []
converged = False
convergence_reason = None

# Actually go round by round
for round_num in range(1, config.max_rounds + 1):
    feedback = ledger.to_prompt_block() if round_num > 1 else None

    review, updated_ledger = await evaluate_round(
        llm_output=first_output,
        context=prompt,
        ledger=ledger,
        round_number=round_num,
        config=config
    )

    ledger = updated_ledger # update with news

    round_result = RoundResult(
        round_number=round_num,
        generated_text=first_output,
        review=review,
        ledger_snapshot=ledger.model_copy(deep=True)
    )
    rounds.append(round_result)

    converged, convergence_reason = check_convergence(review, config, round_num)

    if converged:
        break

best_round = max(rounds, key=lambda r: r.review.overall_score)

final_result = IterativeEvaluationResult(
    rounds=rounds,
    final_output=best_round.generated_text,
    final_score=best_round.review.overall_score,
    converged=converged,
    convergence_reason=convergence_reason
)



In [14]:
from IPython.display import Markdown

Markdown(ledger.to_prompt_block())

=== PRIOR EVALUATION HISTORY ===

Round progression:
  Round 1: Critical factual errors regarding photosynthesis led to low scores across all dimensions.

Open issues (still present — pay attention to these):
  [FC-001] Factual Correctness: Incorrectly states that photosynthesis occurs in the nucleus instead of the chloroplasts.
  [CC-001] Clarity & Communication: Lacks clear structure and detail in the explanation of photosynthesis.
  [DC-001] Depth & Completeness: Fails to include important details about the chemical equation and significance of photosynthesis.
  [RQ-001] Reasoning Quality: Weak reasoning due to inaccuracies and lack of connections between components of photosynthesis.
=================================

In [8]:
print("\n=== RESULTS ===")
for r in final_result.rounds:
    print(f"Round {r.round_number}: {r.review.overall_score}/5.0 ({r.review.letter_grade}) | "
            f"Variance: {r.review.score_variance} | Confidence: {r.review.confidence}")
    print(f"  Open issues: {sum(1 for i in r.ledger_snapshot.issues if i.status == 'open')}")

print(f"\nConverged: {final_result.converged}")
if final_result.convergence_reason:
    print(f"Reason: {final_result.convergence_reason}")
print(f"Final score: {final_result.final_score}/5.0")
print(f"\nBest output:\n{final_result.final_output}")



=== RESULTS ===
Round 1: 2.25/5.0 (D) | Variance: 0.188 | Confidence: low
  Open issues: 4

Converged: False
Final score: 2.25/5.0

Best output:
Photosynthesis is the process by which plants convert sunlight into energy. It occurs in the nucleus of the cell, where chlorophyll captures light and combines water and carbon dioxide to produce glucose and oxygen.
